In [2]:
from google.colab import files

print("Загрузите файл audition.csv")
uploaded = files.upload()

print("Загрузите файл content.csv")
uploaded = files.upload()

import os
print("Файлы после загрузки:", os.listdir('.'))

Загрузите файл audition.csv


Saving audition.csv to audition.csv
Загрузите файл content.csv


Saving content.csv to content.csv
Файлы после загрузки: ['.config', 'audition.csv', 'content.csv', 'sample_data']


In [10]:
from pyspark.sql import SparkSession

# Создаём простую сессию
spark = SparkSession.builder \
    .appName("Bookmate Data Analysis") \
    .getOrCreate()

audition_df = spark.read.csv("audition.csv", inferSchema=True, header=True)
content_df = spark.read.csv("content.csv", inferSchema=True, header=True)

# Проверяем загрузку
print(f"audition: {audition_df.count()} строк")
print(f"content: {content_df.count()} строк")

audition_df.show(10, truncate=False)
content_df.show(10, truncate=False)

audition: 1002895 строк
content: 31667 строк
+-----+---+------------------------------------+---------------+----------+----------+-----+-------------------+-------------------+-----+--------+--------------------+---------+
|162  |0  |68296628-f9d6-11ef-be00-c2c9fa6fd3d5|Станция        |2024-11-26|_c5       |False|0.03777777777777777|0.03777777777777778|True |oCURrBKV|Алматы              |Казахстан|
+-----+---+------------------------------------+---------------+----------+----------+-----+-------------------+-------------------+-----+--------+--------------------+---------+
|213  |1  |682966dc-f9d6-11ef-be00-c2c9fa6fd3d5|Станция        |2024-11-26|NULL      |false|8.333333333333E-4  |0.0                |true |qOL0JJL5|Москва              |Россия   |
|63   |2  |682966dc-f9d6-11ef-be00-c2c9fa6fd3d5|Станция        |2024-11-26|NULL      |false|0.0044444444444444 |0.0                |true |ndM5nzgT|Иркутск             |Россия   |
|2    |4  |68296704-f9d6-11ef-be00-c2c9fa6fd3d5|Станция     

Анализируя данные, я вижу несколько проблем со структурой таблиц.

- Столбцы названы значениями из первой строки данных ("162", "0", "Станция"). Это происходит из-за того, что CSV файлы не имеют корректных заголовков
- Неправильные типы данных, даты распознаны как строки
---
- Таблица audition содержит 1 002 895 строк, данные о пользовательской активности. Каждая строка это одно действие пользователя. Пользователи могут совершать много действий ежедневно.

- Таблица content содержит 31 667 строк, это каталог контента. Каждая строка равна одной книге, аудиокниге, комиксу. Каталог обновляется реже. Количество контента ограничено.

Выводы:
- Один пользователь может прочитать много книг
- Одна книга может быть прочитана многими пользователями
- Активность пользователей содержит значительно больше данных, чем каталог контента

In [35]:
# Загрузка и обработка данных с оптимизациями Spark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Создаём SparkSession
spark = SparkSession.builder \
    .appName("Bookmate Session Analysis") \
    .getOrCreate()


# Переименовываем audition
audition_renamed = audition_df \
    .withColumnRenamed("162", "puid") \
    .withColumnRenamed("0", "audition_id") \
    .withColumnRenamed("68296628-f9d6-11ef-be00-c2c9fa6fd3d5", "session_id") \
    .withColumnRenamed("Станция", "usage_platform_ru") \
    .withColumnRenamed("2024-11-26", "msk_business_dt_str") \
    .withColumnRenamed("_c5", "app_version") \
    .withColumnRenamed("False", "adult_content_flg") \
    .withColumnRenamed("0.03777777777777777", "hours") \
    .withColumnRenamed("0.03777777777777778", "hours_sessions_long") \
    .withColumnRenamed("True", "kids_content_flg") \
    .withColumnRenamed("oCURrBKV", "main_content_id") \
    .withColumnRenamed("Алматы", "usage_geo_city") \
    .withColumnRenamed("Казахстан", "usage_geo_country")

# Переименовываем content_df
content_renamed = content_df \
    .withColumnRenamed(content_df.columns[0], "main_content_id") \
    .withColumnRenamed(content_df.columns[1], "main_content_type") \
    .withColumnRenamed(content_df.columns[2], "main_content_name") \
    .withColumnRenamed(content_df.columns[3], "main_content_duration_hours") \
    .withColumnRenamed(content_df.columns[4], "published_topic_title_list") \
    .withColumnRenamed(content_df.columns[5], "main_author_id")

# Создаем DataFrame с нужными столбцами
result_df = (audition_renamed
    .select("puid", "hours_sessions_long", "msk_business_dt_str", "adult_content_flg")
    .withColumn("minutes_sessions_long", (F.col("hours_sessions_long") * 60).cast("int"))
    .withColumn("date", F.to_date("msk_business_dt_str"))
    .withColumn("day_of_week", F.dayofweek("date"))
    .withColumn("is_weekend", F.when((F.col("day_of_week") == 1) | (F.col("day_of_week") == 7), True).otherwise(False))
    .select("puid", "hours_sessions_long", "minutes_sessions_long", "is_weekend", "adult_content_flg")
)

# Выводим первые десять строк полученной таблицы
print("Первые 10 строк таблицы:")
result_df.show(10, truncate=False)

# Избавляемся от пропусков в adult_content_flg
result_df = result_df.filter(F.col("adult_content_flg").isNotNull())

# Рассчитываем суммы минут по выходным/будним и взрослому/невзрослому контенту
summary_df = (result_df
    .groupBy("adult_content_flg", "is_weekend")
    .agg(
        F.sum("minutes_sessions_long").alias("total_minutes"),
        F.avg("minutes_sessions_long").alias("avg_minutes_per_session"),
        F.count("minutes_sessions_long").alias("sessions_count")
    )
    .orderBy("adult_content_flg", "is_weekend")  # Сначала по возрастному рейтингу, затем по выходным
)

print("\nАнализ времени сессий по возрастному рейтингу и типам дней:")
summary_df.show(truncate=False)

Первые 10 строк таблицы:
+-----+-------------------+---------------------+----------+-----------------+
|puid |hours_sessions_long|minutes_sessions_long|is_weekend|adult_content_flg|
+-----+-------------------+---------------------+----------+-----------------+
|213  |0.0                |0                    |false     |false            |
|63   |0.0                |0                    |false     |false            |
|2    |0.0                |0                    |false     |true             |
|28   |0.0                |0                    |false     |false            |
|38   |0.4890794444444444 |29                   |false     |true             |
|11162|0.1847275          |11                   |false     |true             |
|11119|1.4530555555555555 |87                   |false     |true             |
|47   |0.5399818181818182 |32                   |false     |true             |
|194  |0.0                |0                    |false     |true             |
|213  |0.2905090909090909 |

1. Различие по возрастному рейтингу
Это самый значительный фактор, влияющий на поведение.

- Контент для взрослых: средняя продолжительность сессии примерно 53 минуты. Это очень высокий показатель, указывающий на вовлечение пользователей.

- Обычный контент: средняя продолжительность сессии примерно 7,7 минут. Это типично для быстрого потребления информации или коротких развлекательных видео (как в соцсетях).

Вывод: Разница в среднем времени сессии более чем в 6,5 раз между двумя группами говорит о принципиально разных моделях использования платформы.

2. Влияние дня недели
Выходные дни оказывают слабое, но заметное влияние на поведение. В выходные дни средняя продолжительность сессии незначительно увеличивается в обеих группах.

- Для обычного контента: с 7,66 до 7,81 мин. (+0,15 мин).

- Для контента для взрослых: с 53 до 53,42 мин. (+0,42 мин.).

Вывод: У людей больше свободного времени в выходные, что позволяет им проводить за сессией чуть дольше. Однако это влияние минимально по сравнению с фактором возрастного рейтинга.

3. Анализ общего времени и количества сессий
Здесь диспропорция.

- Обычный контент: количество сессий: 445888 в будни + 181809 в выходные = 627697. Это значительно больше, чем у взрослой группы. Платформа активно используется для коротких сессий.

- Контент для взрослых: Количество сессий: 277372  в будни + 97827 в выходные = 375199.

Вывод: Аудитория контента для взрослых генерирует непропорционально большой объем общего времени потребления благодаря экстремально высокой продолжительности каждой сессии.


In [37]:
# Объединяем таблицы
joined_df = audition_renamed.join(content_renamed, "main_content_id", "inner")

# Удаляем указанные столбцы
columns_to_drop = ["main_author_id", "app_version"]
existing_columns_to_drop = [col for col in columns_to_drop if col in joined_df.columns]
cleaned_df = joined_df.drop(*existing_columns_to_drop)

print("\nПервые 10 строк объединенной таблицы после удаления столбцов:")
cleaned_df.show(10, truncate=False)

# Считаем количество уникальных пользователей
unique_users_joined = cleaned_df.select("puid").distinct().count()
unique_users_audition = audition_renamed.select("puid").distinct().count()

print(f"- В объединенной таблице: {unique_users_joined}")
print(f"- В исходной таблице audition_df: {unique_users_audition}")

# Анализ типов контента
print(f"\nС помощью collect() получены все уникальные типы контента:")

# Считаем общее количество записей
total_records = cleaned_df.count()

# Группируем по типам контента
content_distribution = cleaned_df.groupBy("main_content_type") \
    .agg(
        F.count("*").alias("records_count")
    ) \
    .withColumn(
        "percentage",
        F.round(F.col("records_count") / total_records * 100, 1)
    ) \
    .orderBy(F.desc("records_count"))

# Используем collect() для получения всех строк
distribution_list = content_distribution.collect()

# Выводим результаты
for row in distribution_list:
    eng_type = row["main_content_type"]
    count = row["records_count"]
    percent = row["percentage"]

    if eng_type == "Audiobook":
        print(f"Audiobook (Аудиокниги) - {count} записей ({percent}%)")
    elif eng_type == "Book":
        print(f"Book (Книги) - {count} записей ({percent}%)")
    elif eng_type == "Comicbook":
        print(f"Comicbook (Комиксы) - {count} записей ({percent}%)")



Первые 10 строк объединенной таблицы после удаления столбцов:
+---------------+-----+-----------+------------------------------------+-----------------+-------------------+-----------------+------------------+-------------------+----------------+--------------------+-----------------+-----------------+---------------------------------------------------------------------------------------------------------+---------------------------+----------------------------------------------------------------------------------------------------+
|main_content_id|puid |audition_id|session_id                          |usage_platform_ru|msk_business_dt_str|adult_content_flg|hours             |hours_sessions_long|kids_content_flg|usage_geo_city      |usage_geo_country|main_content_type|main_content_name                                                                                        |main_content_duration_hours|published_topic_title_list                                                           

1. Объединение таблиц по main_content_id

Количество строк после объединения - 996534 строк, соответствует ожиданиям.

2. Удаление лишних столбцов

Удалены столбцы, не нужные для анализа: main_author_id, app_version

3. Анализ пользовательской базы

Уникальные пользователи в объединенной таблице = 2574
Уникальные пользователи в исходной audition_df =  2576
Разница = 2 пользователя, которые использовали контент, отсутствующий в таблице content_df

4. Уникальные значения main_content_type

С помощью collect() получены все уникальные типы контента:

- Audiobook (Аудиокниги) - 891770 записей (89,4%)
- Book (Книги) - 97774 записей (9,8%)
- Comicbook (Комиксы) - 6990 записей (0,7%)